# Generate Predictions with Top-K Retrieval

**Mục tiêu**: Tạo predictions cho evaluation, lưu **TOP 100** items có similarity cao nhất cho mỗi query

**Quy trình**:
1. Load 200 query_id từ ground truth JSONL
2. Load tất cả models (TFIDF, Ingredient_TFIDF, Keyword, Hybrid, SBERT, Hybrid_TFIDF_SBERT)
3. Với mỗi query_id:
   - Tính similarity scores với tất cả items trong candidate pool (10k recipes)
   - Sắp xếp theo score giảm dần (exclude self)
   - Lấy top 100 items
4. Lưu predictions vào `evaluation_jsonl/<method>_pred.jsonl` với format:
   ```json
   {"query_id": 123, "relevant_docs": [{"doc_id": 456, "score": 0.85}, ...]}
   ```

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import pickle
import json
import os
from pathlib import Path
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sentence_transformers import SentenceTransformer
import faiss

e:\DS300-UIT-RecommenderSystem\DS300-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setup paths
DATA_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/data/all_recipes_final.csv"
MODELS_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/Saved_models"
GROUND_TRUTH_PATH = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/eval_ground_truth.jsonl"
OUTPUT_DIR = r"E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl"

# Number of top items to retrieve per query
TOP_K = 100

# Create output directory if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Data path: {DATA_PATH}")
print(f"Models path: {MODELS_PATH}")
print(f"Ground truth path: {GROUND_TRUTH_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Top K: {TOP_K}")

Data path: E:\DS300-UIT-RecommenderSystem/Finalproject/data/all_recipes_final.csv
Models path: E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/Saved_models
Ground truth path: E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/eval_ground_truth.jsonl
Output directory: E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl
Top K: 100


## 1. Load Ground Truth and Data

In [3]:
# Ground truth (json)
ground_truth = []
with open(GROUND_TRUTH_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        ground_truth.append(json.loads(line.strip()))

In [4]:
# query_ids (list)
query_ids = [item['query_id'] for item in ground_truth]
print(f"Loaded {len(query_ids)} query_ids from ground truth")
print(f"First 5 query_ids: {query_ids[:5]}")

Loaded 200 query_ids from ground truth
First 5 query_ids: [49, 84, 140, 191, 246]


In [5]:
# Load data (all recipes - 10k candidate pool)
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} recipes")
print(f"Columns: {df.columns.tolist()}")

Loaded 10263 recipes
Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'ingredients_normalized', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source']


In [6]:
# Add recipe_id column (using index as recipe_id)
df['recipe_id'] = df.index
print(f"Columns: {df.columns.tolist()}")

Columns: ['title', 'type_of_food', 'link', 'description', 'ingredients', 'ingredients_normalized', 'step', 'note', 'num_of_ingredients', 'cook_time', 'num_of_people', 'calories', 'source', 'recipe_id']


## 2. Load All Models

In [7]:
# Method 1: TFIDF (text-based: title + description + steps)
with open(os.path.join(MODELS_PATH, "TFIDF", "tfidf_vectorizer.pkl"), 'rb') as f:
    tfidf_vectorizer = pickle.load(f)
    
tfidf_similarity = np.load(os.path.join(MODELS_PATH, "TFIDF", "tfidf_similarity.npy"))

with open(os.path.join(MODELS_PATH, "TFIDF", "tfidf_processed_data.pkl"), 'rb') as f:
    tfidf_data = pickle.load(f)

print(f"TFIDF similarity matrix shape: {tfidf_similarity.shape}")
print(f"TFIDF data shape: {len(tfidf_data)}")

TFIDF similarity matrix shape: (10263, 10263)
TFIDF data shape: 10263


In [8]:
# Method 2: Ingredient_TFIDF (ingredient-based)
with open(os.path.join(MODELS_PATH, "Ingredient_TFIDF", "ingredient_tfidf_vectorizer.pkl"), 'rb') as f:
    ingredient_tfidf_vectorizer = pickle.load(f)
    
ingredient_tfidf_similarity = np.load(os.path.join(MODELS_PATH, "Ingredient_TFIDF", "ingredient_tfidf_similarity.npy"))

print(f"Ingredient TFIDF similarity matrix shape: {ingredient_tfidf_similarity.shape}")

Ingredient TFIDF similarity matrix shape: (10263, 10263)


In [9]:
# Method 3: Keyword (TF-IDF on keywords extracted from title)
keyword_similarity = np.load(os.path.join(MODELS_PATH, "Keyword", "keyword_similarity.npy"))

print(f"Keyword similarity matrix shape: {keyword_similarity.shape}")

Keyword similarity matrix shape: (10263, 10263)


In [10]:
# Method 4: Hybrid (combine text_tfidf + ingredient_tfidf)
hybrid_similarity = np.load(os.path.join(MODELS_PATH, "Hybrid", "hybrid_similarity.npy"))

print(f"Hybrid similarity matrix shape: {hybrid_similarity.shape}")

Hybrid similarity matrix shape: (10263, 10263)


In [11]:
# Method 5: SBERT + FAISS (Semantic embeddings)

# Load model info
sbert_dir = os.path.join(MODELS_PATH, "SBERT_FAISS")
with open(os.path.join(sbert_dir, "model_info.json"), 'r', encoding='utf-8') as f:
    sbert_info = json.load(f)

# Load SBERT model
sbert_model = SentenceTransformer(sbert_info['model_name'])
print(f"  Loaded SBERT model: {sbert_info['model_name']}")

# Load recipe embeddings
sbert_embeddings = np.load(os.path.join(sbert_dir, "recipe_embeddings.npy"))
print(f"  SBERT embeddings shape: {sbert_embeddings.shape}")

# Load FAISS index
faiss_index = faiss.read_index(os.path.join(sbert_dir, "faiss_index.bin"))
print(f"  FAISS index loaded: {faiss_index.ntotal} vectors")

  Loaded SBERT model: keepitreal/vietnamese-sbert
  SBERT embeddings shape: (10263, 768)
  FAISS index loaded: 10263 vectors


In [12]:
# Compute SBERT similarity matrix
print("Computing SBERT similarity matrix...")
sbert_similarity = cosine_similarity(sbert_embeddings, sbert_embeddings)
print(f"  SBERT similarity matrix shape: {sbert_similarity.shape}")

Computing SBERT similarity matrix...
  SBERT similarity matrix shape: (10263, 10263)


In [14]:
# Method 6: Hybrid TF-IDF + SBERT (Ensemble)
print("Loading Hybrid TF-IDF + SBERT model...")

# Load config
hybrid_dir = os.path.join(MODELS_PATH, "Hybrid_TFIDF_SBERT")
with open(os.path.join(hybrid_dir, "config.json"), 'r', encoding='utf-8') as f:
    hybrid_config = json.load(f)

# Load SBERT embeddings (already loaded above, but load from hybrid dir for completeness)
hybrid_sbert_embeddings = np.load(os.path.join(hybrid_dir, "sbert_embeddings.npy"))
print(f"  Hybrid SBERT embeddings shape: {hybrid_sbert_embeddings.shape}")

# TF-IDF components are already loaded (reusing from TFIDF method)
print(f"  Reusing TF-IDF from method 1")
print(f"  Alpha (weight for TF-IDF): {hybrid_config['alpha']}")

# Compute hybrid similarity matrix (alpha * TF-IDF + (1-alpha) * SBERT)
alpha = hybrid_config['alpha']
print(f"Computing Hybrid similarity matrix (alpha={alpha})...")

# Normalize TF-IDF similarity to [0, 1] range
tfidf_sim_normalized = tfidf_similarity.copy()
# TF-IDF cosine similarity is already in [-1, 1], typically [0, 1] for non-negative vectors

# SBERT similarity is already normalized (cosine on normalized embeddings)
# Combine: alpha * TF-IDF + (1-alpha) * SBERT
hybrid_tfidf_sbert_similarity = alpha * tfidf_sim_normalized + (1 - alpha) * sbert_similarity
print(f"  Hybrid similarity matrix shape: {hybrid_tfidf_sbert_similarity.shape}")

Loading Hybrid TF-IDF + SBERT model...
  Hybrid SBERT embeddings shape: (10263, 768)
  Reusing TF-IDF from method 1
  Alpha (weight for TF-IDF): 0.5
Computing Hybrid similarity matrix (alpha=0.5)...
  Hybrid similarity matrix shape: (10263, 10263)


## 3. Implement Top-K Retrieval Function

In [15]:
def retrieve_top_k(query_idx, similarity_matrix, top_k=100, exclude_self=True):
    """
    Retrieve top K most similar items for a given query
    
    Args:
        query_idx: index of query item in dataframe
        similarity_matrix: precomputed similarity matrix
        top_k: number of top items to retrieve
        exclude_self: whether to exclude query item itself
    
    Returns:
        list of (doc_id, score) tuples, sorted by score descending
    """
    # Get similarity scores for query
    scores = similarity_matrix[query_idx].copy()
    
    # Exclude self if requested
    if exclude_self:
        scores[query_idx] = -1  # Set to -1 to exclude from results
    
    # Get top K indices (sorted by score descending)
    top_indices = np.argsort(scores)[::-1][:top_k]
    
    # Get doc_ids and scores
    results = []
    for idx in top_indices:
        doc_id = df.iloc[idx]['recipe_id']
        score = float(scores[idx])
        results.append({"doc_id": int(doc_id), "score": score})
    
    return results

In [16]:
# Test function with different K values
test_query_idx = 0
print(f"Test query (index {test_query_idx}):")
print(f"Query recipe_id: {df.iloc[test_query_idx]['recipe_id']}")
print(f"Query title: {df.iloc[test_query_idx]['title']}")
print()

Test query (index 0):
Query recipe_id: 0
Query title: Cách muối dưa hành truyền thống



In [17]:
for k in [10, 50, 100]:
    test_results = retrieve_top_k(test_query_idx, tfidf_similarity, top_k=k)
    print(f"Top {k}: Retrieved {len(test_results)} items")
    if len(test_results) > 0:
        print(f"  Top 3 scores: [{test_results[0]['score']:.4f}, {test_results[1]['score']:.4f}, {test_results[2]['score']:.4f}]")
print()

Top 10: Retrieved 10 items
  Top 3 scores: [0.6396, 0.4544, 0.3689]
Top 50: Retrieved 50 items
  Top 3 scores: [0.6396, 0.4544, 0.3689]
Top 100: Retrieved 100 items
  Top 3 scores: [0.6396, 0.4544, 0.3689]



In [18]:
# Use default TOP_K
test_results = retrieve_top_k(test_query_idx, tfidf_similarity, top_k=TOP_K)
print(f"Top {TOP_K}: Retrieved {len(test_results)} items")
print(f"Sample results (top 5):")
for i, item in enumerate(test_results[:5], 1):
    doc_row = df[df['recipe_id'] == item['doc_id']].iloc[0]
    print(f"  {i}. Doc {item['doc_id']} (score: {item['score']:.4f}) - {doc_row['title']}")

Top 100: Retrieved 100 items
Sample results (top 5):
  1. Doc 52 (score: 0.6396) - Cách muối hành trắng giòn, để được lâu
  2. Doc 9321 (score: 0.4544) - Cách muối dưa hành giòn ngon, không bị hăng đơn giản tại nhà
  3. Doc 10057 (score: 0.3689) - Cách muối dưa củ cải giòn ngon không hăng bằng hộp đựng thực phẩm
  4. Doc 8290 (score: 0.3686) - Cách ngâm hành tím thái lát ăn liền ăn bún bò Huế chuẩn vị
  5. Doc 619 (score: 0.3302) - Dưa góp kiểu miền Trung


## 4. Run Evaluation Loop for All Methods

In [19]:
def run_evaluation(method_name, similarity_matrix, query_ids, df, top_k=100):
    """
    Run evaluation for a specific method using top-K retrieval
    
    Args:
        method_name: name of the method (for output file)
        similarity_matrix: precomputed similarity matrix
        query_ids: list of query recipe_ids
        df: dataframe containing all recipes
        top_k: number of top items to retrieve per query
    
    Returns:
        list of predictions (one per query)
    """
    predictions = []
    
    # Create recipe_id to index mapping
    recipe_id_to_idx = {recipe_id: idx for idx, recipe_id in enumerate(df['recipe_id'])}
    
    for query_id in tqdm(query_ids, desc=f"Evaluating {method_name}"):
        # Get query index
        query_idx = recipe_id_to_idx[query_id]
        
        # Retrieve top K items (excluding self)
        top_items = retrieve_top_k(
            query_idx, 
            similarity_matrix, 
            top_k=top_k, 
            exclude_self=True
        )
        
        # Create prediction record
        pred_record = {
            "query_id": int(query_id),
            "relevant_docs": top_items  # Top K items
        }
        predictions.append(pred_record)
    
    # Print statistics
    print(f"\nStatistics for {method_name}:")
    print(f"  Total queries: {len(predictions)}")
    print(f"  Items per query: {top_k}")
    print(f"  Total predictions: {len(predictions) * top_k}")
    
    return predictions

In [20]:
# Test with small subset first
test_predictions = run_evaluation("test", tfidf_similarity, query_ids[:5], df, top_k=TOP_K)
print(f"\nExample prediction:")
print(f"Query ID: {test_predictions[0]['query_id']}")
print(f"Number of recommended items: {len(test_predictions[0]['relevant_docs'])}")
print(f"Top 5 recommendations: {test_predictions[0]['relevant_docs'][:5]}")

Evaluating test: 100%|██████████| 5/5 [00:00<00:00, 67.40it/s]


Statistics for test:
  Total queries: 5
  Items per query: 100
  Total predictions: 500

Example prediction:
Query ID: 49
Number of recommended items: 100
Top 5 recommendations: [{'doc_id': 9042, 'score': 0.8336782523772797}, {'doc_id': 69, 'score': 0.7967552382186476}, {'doc_id': 9150, 'score': 0.7818251200729153}, {'doc_id': 8863, 'score': 0.7794561964558814}, {'doc_id': 9651, 'score': 0.7791967501699464}]


In [21]:
# Run evaluation for all 6 methods with top-K retrieval
methods = [
    ("TFIDF", tfidf_similarity),
    ("Ingredient_TFIDF", ingredient_tfidf_similarity),
    ("Keyword", keyword_similarity),
    ("Hybrid", hybrid_similarity),
    ("SBERT_FAISS", sbert_similarity),
    ("Hybrid_TFIDF_SBERT", hybrid_tfidf_sbert_similarity)
]

all_predictions = {}

print(f"Running evaluation with TOP_K = {TOP_K}")
print(f"This will retrieve top {TOP_K} most similar items for each query")
print(f"Total methods: {len(methods)}")

Running evaluation with TOP_K = 100
This will retrieve top 100 most similar items for each query
Total methods: 6


In [22]:
for method_name, similarity_matrix in methods:
    print(f"\n{'='*80}")
    print(f"Method: {method_name}")
    print(f"{'='*80}")
    
    # Run evaluation with top-K
    predictions = run_evaluation(
        method_name, 
        similarity_matrix, 
        query_ids, 
        df, 
        top_k=TOP_K
    )
    all_predictions[method_name] = predictions
    
    # Save to JSONL file
    output_file = os.path.join(OUTPUT_DIR, f"{method_name}_pred.jsonl")
    with open(output_file, 'w', encoding='utf-8') as f:
        for pred in predictions:
            f.write(json.dumps(pred, ensure_ascii=False) + '\n')
    
    print(f"Saved {len(predictions)} predictions to {output_file}")
    print(f"  Total predictions: {sum(len(p['relevant_docs']) for p in predictions)}")


Method: TFIDF


Evaluating TFIDF: 100%|██████████| 200/200 [00:01<00:00, 160.20it/s]



Statistics for TFIDF:
  Total queries: 200
  Items per query: 100
  Total predictions: 20000
Saved 200 predictions to E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl\TFIDF_pred.jsonl
  Total predictions: 20000

Method: Ingredient_TFIDF


Evaluating Ingredient_TFIDF: 100%|██████████| 200/200 [00:01<00:00, 164.29it/s]



Statistics for Ingredient_TFIDF:
  Total queries: 200
  Items per query: 100
  Total predictions: 20000
Saved 200 predictions to E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl\Ingredient_TFIDF_pred.jsonl
  Total predictions: 20000

Method: Keyword


...


Statistics for Hybrid:
  Total queries: 200
  Items per query: 100
  Total predictions: 20000
Saved 200 predictions to E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl\Hybrid_pred.jsonl
  Total predictions: 20000

Method: SBERT_FAISS


Evaluating SBERT_FAISS: 100%|██████████| 200/200 [00:01<00:00, 184.98it/s]



Statistics for SBERT_FAISS:
  Total queries: 200
  Items per query: 100
  Total predictions: 20000
Saved 200 predictions to E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl\SBERT_FAISS_pred.jsonl
  Total predictions: 20000

Method: Hybrid_TFIDF_SBERT


Evaluating Hybrid_TFIDF_SBERT: 100%|██████████| 200/200 [00:01<00:00, 199.33it/s]


Statistics for Hybrid_TFIDF_SBERT:
  Total queries: 200
  Items per query: 100
  Total predictions: 20000
Saved 200 predictions to E:\DS300-UIT-RecommenderSystem/Finalproject/notebooks/recommend_and_evaluation/evaluation_jsonl\Hybrid_TFIDF_SBERT_pred.jsonl
  Total predictions: 20000


In [23]:
# Summary
print(f"\n{'='*80}")
print("SUMMARY - Total predictions per method:")
print(f"{'='*80}")
for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid", "SBERT_FAISS", "Hybrid_TFIDF_SBERT"]:
    total_predictions = sum(len(p['relevant_docs']) for p in all_predictions[method_name])
    avg_per_query = total_predictions / len(query_ids)
    print(f"  {method_name:25s}: {total_predictions:6d} total predictions ({avg_per_query:6.2f} avg per query)")


SUMMARY - Total predictions per method:
  TFIDF                    :  20000 total predictions (100.00 avg per query)
  Ingredient_TFIDF         :  20000 total predictions (100.00 avg per query)
  Keyword                  :  20000 total predictions (100.00 avg per query)
  Hybrid                   :  20000 total predictions (100.00 avg per query)
  SBERT_FAISS              :  20000 total predictions (100.00 avg per query)
  Hybrid_TFIDF_SBERT       :  20000 total predictions (100.00 avg per query)


## 6. Sanity Checks

### 6.1. Verify no self-recommendations

In [24]:
# Verify that no prediction contains the query_id itself
def verify_no_self_recommendations(predictions):
    """Check if any prediction contains self-recommendation"""
    violations = []
    
    for pred in predictions:
        query_id = pred['query_id']
        doc_ids = [item['doc_id'] for item in pred['relevant_docs']]
        
        if query_id in doc_ids:
            violations.append(query_id)
    
    return violations

In [25]:
for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid", "SBERT_FAISS", "Hybrid_TFIDF_SBERT"]:
    preds = all_predictions[method_name]
    violations = verify_no_self_recommendations(preds)
    
    if violations:
        print(f"{method_name}: Found {len(violations)} self-recommendations!")
        print(f"   Query IDs with self-recommendation: {violations[:5]}...")
    else:
        print(f"{method_name}: No self-recommendations found ({len(preds)} queries checked)")

TFIDF: No self-recommendations found (200 queries checked)
Ingredient_TFIDF: No self-recommendations found (200 queries checked)
Keyword: No self-recommendations found (200 queries checked)
Hybrid: No self-recommendations found (200 queries checked)
SBERT_FAISS: No self-recommendations found (200 queries checked)
Hybrid_TFIDF_SBERT: No self-recommendations found (200 queries checked)


### 6.2. Check score distributions

In [26]:
for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid", "SBERT_FAISS", "Hybrid_TFIDF_SBERT"]:
    preds = all_predictions[method_name]
    sizes = [len(p['relevant_docs']) for p in preds]
    
    print(f"\n{method_name}:")
    print(f"  Items per query: {sizes[0]} (should all be {TOP_K})")
    print(f"  Total queries: {len(sizes)}")
    print(f"  Consistent: {all(s == TOP_K for s in sizes)}")


TFIDF:
  Items per query: 100 (should all be 100)
  Total queries: 200
  Consistent: True

Ingredient_TFIDF:
  Items per query: 100 (should all be 100)
  Total queries: 200
  Consistent: True

Keyword:
  Items per query: 100 (should all be 100)
  Total queries: 200
  Consistent: True

Hybrid:
  Items per query: 100 (should all be 100)
  Total queries: 200
  Consistent: True

SBERT_FAISS:
  Items per query: 100 (should all be 100)
  Total queries: 200
  Consistent: True

Hybrid_TFIDF_SBERT:
  Items per query: 100 (should all be 100)
  Total queries: 200
  Consistent: True


### 6.3. Score distribution analysis

In [27]:
for method_name in ["TFIDF", "Ingredient_TFIDF", "Keyword", "Hybrid", "SBERT_FAISS", "Hybrid_TFIDF_SBERT"]:
    preds = all_predictions[method_name]
    all_scores = []
    for pred in preds:
        all_scores.extend([item['score'] for item in pred['relevant_docs']])
    
    if len(all_scores) > 0:
        print(f"\n{method_name} (total {len(all_scores)} predictions):")
        print(f"  Score range: [{min(all_scores):.4f}, {max(all_scores):.4f}]")
        print(f"  Mean: {np.mean(all_scores):.4f}")
        print(f"  Median: {np.median(all_scores):.4f}")
        print(f"  Std: {np.std(all_scores):.4f}")
        
        # Percentiles
        percentiles = [25, 50, 75, 90, 95, 99]
        print(f"  Score percentiles:")
        for p in percentiles:
            val = np.percentile(all_scores, p)
            print(f"    {p}th: {val:.4f}")


TFIDF (total 20000 predictions):
  Score range: [0.1106, 0.9794]
  Mean: 0.4204
  Median: 0.4105
  Std: 0.1271
  Score percentiles:
    25th: 0.3348
    50th: 0.4105
    75th: 0.5082
    90th: 0.5867
    95th: 0.6370
    99th: 0.7354

Ingredient_TFIDF (total 20000 predictions):
  Score range: [0.1255, 1.0000]
  Mean: 0.3157
  Median: 0.3039
  Std: 0.0829
  Score percentiles:
    25th: 0.2609
    50th: 0.3039
    75th: 0.3564
    90th: 0.4148
    95th: 0.4596
    99th: 0.5792

Keyword (total 20000 predictions):
  Score range: [0.0385, 0.5714]
  Mean: 0.1890
  Median: 0.1875
  Std: 0.0611
  Score percentiles:
    25th: 0.1471
    50th: 0.1875
    75th: 0.2258
    90th: 0.2647
    95th: 0.2917
    99th: 0.3600

Hybrid (total 20000 predictions):
  Score range: [0.1186, 0.9918]
  Mean: 0.3189
  Median: 0.3105
  Std: 0.0789
  Score percentiles:
    25th: 0.2664
    50th: 0.3105
    75th: 0.3633
    90th: 0.4152
    95th: 0.4524
    99th: 0.5500

SBERT_FAISS (total 20000 predictions):
  Scor